## Write Urself Challenge
this is a section where I have to rewrite everything from memory and figure it out myself

In [61]:
import math
import random

In [62]:
class Value:
    def __init__(self, num, _child=(), label=''):
        self.data   = num
        self.label  = label

        self._prev  = set(_child)
        self._backward = lambda: None
        self.grad   = 0.0

    # print
    def __repr__(self):
        return f"Value({self.label}:{self.data})"


    # basic operations
    def __add__ (self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))

        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad 
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))

        def _backward():
            self.grad  += other.data * out.grad
            other.grad +=  self.data * out.grad 
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,))

        def _backward():
            self.grad += other * (self.data**(other-1)) * out.grad
        out._backward = _backward

        return out

    def __truediv__(self, other):
        return self * other**-1

    def __neg__(self):
        return self * -1 

    def __sub__(self, other):
        return self + (-other)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def exp(self):
        out = Value(math.exp(self.data), (self,))

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward

        return out

    # activation func
    def sigmoid(self):
        out = Value(1 / (1 + (-self).exp().data), (self,))

        def _backward():
            self.grad += out.data * (1 - out.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self,))

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward

        return out

    def relu(self):
        t = max(0, self.data)
        out = Value(t, (self,))
        
        def _backward():
            self.grad += (1 if t > 0 else 0) * out.grad
        out._backward = _backward

        return out

    # backprop
    def backward(self):
        topo=[]
        vis =set()

        def build_topo(val):
            if val not in vis:
                vis.add(val)
                for c in val._prev:     
                    build_topo(c)
                topo.append(val)

        build_topo(self)
        self.grad = 1.0
        for val in reversed(topo):
            val._backward()

In [59]:
a = Value(4.0, label='a');
b = Value(3.0, label='b')
c = Value(1.0, label='c')

d = a * b; d.label='d'
e = d + c; e.label='e'
s = e.relu()

s.backward()
s


Value(:13.0)

In [60]:
print(s.grad)
print(e.grad)
print(d.grad)
print(c.grad)
print(b.grad)
print(a.grad)

1.0
1.0
1.0
1.0
4.0
3.0


### NEURAL NETWORK I AM COMING BABYYYY

In [ ]:
class Neuron:
    def __init__(self, nin, activation=None):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b =  Value(random.uniform(-1,1))

        acts = {
            'relu': lambda x: x.relu(),
            'sigmoid': lambda x: x.sigmoid(),
            'tanh': lambda x: x.tanh(),
            'identity': lambda x: x,
        }
        self.activation = acts[activation]

    def __call__(self, x):
        out = sum((xi*wi for xi, wi in zip(x, self.w)), self.b)
        return self.activation(out)

    def parameters(self):
        return [self.b] + self.w

In [106]:
class Layer:
    def __init__(self, nin, nout, activation=None):
        self.neurons = [Neuron(nin, activation=activation) for _ in range(nout)] 

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

In [110]:
class MLP:
    def __init__(self, nin, nouts):
        self.layers = []
        prev = nin

        for nout, act in nouts:
            self.layers.append(Layer(nin, nout, activation=act))
            prev = nout

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [276]:
x = [2.0, 3.0, 4.0]

n = MLP(len(x), [
    (3, 'relu'),
    (4, 'relu'),
    (1, 'sigmoid')
    ])

n(x)

Value(:0.8283299856792645)

In [ ]:
# 8 samples with 4 inputs each
xs = [
    [ 1.5,  2.0,  1.0,  0.5], # (1.5*2.0) - (1.0^2) + 0.5 = 3.0 - 1.0 + 0.5 = +2.5 -> 1.0
    [ 2.0, -1.0,  1.5, -0.5], # (2.0*-1.0) - (1.5^2) - 0.5 = -2.0 - 2.25 - 0.5 = -4.75 -> 0.0
    [-1.0, -2.0,  1.0, -0.5], # (-1.0*-2.0) - (1.0^2) - 0.5 = 2.0 - 1.0 - 0.5 = +0.5 -> 1.0
    [ 0.5,  1.0,  2.0,  1.0], # (0.5*1.0) - (2.0^2) + 1.0 = 0.5 - 4.0 + 1.0 = -2.5 -> 0.0
    [ 2.0,  2.0,  0.5, -1.0], # (2.0*2.0) - (0.5^2) - 1.0 = 4.0 - 0.25 - 1.0 = +2.75 -> 1.0
    [-1.5,  1.0,  1.0,  0.0], # (-1.5*1.0) - (1.0^2) + 0.0 = -1.5 - 1.0 + 0.0 = -2.5 -> 0.0
    [ 1.0,  3.0,  1.2, -0.2], # (1.0*3.0) - (1.2^2) - 0.2 = 3.0 - 1.44 - 0.2 = +1.36 -> 1.0
    [-2.0,  0.5,  1.5, -1.0], # (-2.0*0.5) - (1.5^2) - 1.0 = -1.0 - 2.25 - 1.0 = -4.25 -> 0.0
]

ys = [1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0]

In [279]:
# 4 inputs, 2 hidden layers with ReLU, 1 output neuron with Sigmoid
n = MLP(4, [
    (6, 'relu'),
    (6, 'relu'),
    (1, 'sigmoid')
])

In [ ]:
lr = 0.001

ypred = [n(x) for x in xs]
loss = sum((yout - ygt)**2 for yout, ygt in zip(ypred, ys))
loss.backward()

print(n.parameters())

for p in n.parameters():
    p.data += -lr * p.grad

print("=======")
print(n.parameters())
print(loss)

[Value(:0.9313406593219334), Value(:-0.5205263994742191), Value(:-1.6553176119940418), Value(:-1.373451726187621), Value(:-0.07812899706410688), Value(:-0.77290983724255), Value(:0.5078024509274828), Value(:-0.49953787815353357), Value(:-0.6533877735839324), Value(:-0.36523627210062015), Value(:-1.088101580740334), Value(:-0.17132515431071685), Value(:-0.0449672438672885), Value(:0.3108173629647695), Value(:0.3965134192736347), Value(:-0.8441621498072291), Value(:3.132643580408316), Value(:2.4128240442661135), Value(:-2.1113406334112605), Value(:0.09988688061411113), Value(:0.49687005730508216), Value(:0.8207756444840493), Value(:0.6742782133826917), Value(:-0.9890349368537754), Value(:0.4109182938199003), Value(:-0.8685697848900304), Value(:-0.4289733358853387), Value(:-0.4212043008331383), Value(:0.1725722961645928), Value(:0.8135147403614775), Value(:-0.4526960806088802), Value(:-0.14873162980728494), Value(:-0.25783069438043227), Value(:0.3913585790138807), Value(:-0.27781384812581

In [417]:
ypred = [n(x) for x in xs]

for ygt, yout in zip(ys, ypred):
    prob = yout.data
    pred = 1.0 if prob >= 0.5 else 0
    print(f"Target: {ygt} | Prob: {prob:.4f} | Predicted: {pred}")

Target: 1.0 | Prob: 1.0000 | Predicted: 1.0
Target: 0.0 | Prob: 0.0067 | Predicted: 0
Target: 1.0 | Prob: 0.9977 | Predicted: 1.0
Target: 0.0 | Prob: 0.0067 | Predicted: 0
Target: 1.0 | Prob: 1.0000 | Predicted: 1.0
Target: 0.0 | Prob: 0.0067 | Predicted: 0
Target: 1.0 | Prob: 1.0000 | Predicted: 1.0
Target: 0.0 | Prob: 0.0067 | Predicted: 0
